In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries loaded!')

✅ Libraries loaded!


In [5]:
results = pd.read_csv('results.csv')

print(f'Shape: {results.shape}')
print(f'Columns: {list(results.columns)}')
print(f'\nDate range: {results["date"].min()} to {results["date"].max()}')

Shape: (49498, 9)
Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']

Date range: 1872-11-30 to 2026-07-06


In [6]:
tournament_counts = results['tournament'].value_counts()
print(tournament_counts.head(20))

tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1057
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Gold Cup                                  420
Gulf Cup                                  410
Island Games                              394
UEFA Euro                                 388
Asian Games                               368
Name: count, dtype: int64


In [7]:
results.head(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
5,1876-03-25,Scotland,Wales,4.0,0.0,Friendly,Glasgow,Scotland,False
6,1877-03-03,England,Scotland,1.0,3.0,Friendly,London,England,False
7,1877-03-05,Wales,Scotland,0.0,2.0,Friendly,Wrexham,Wales,False
8,1878-03-02,Scotland,England,7.0,2.0,Friendly,Glasgow,Scotland,False
9,1878-03-23,Scotland,Wales,9.0,0.0,Friendly,Glasgow,Scotland,False


In [8]:
# Filter to post-1990 and remove friendlies
results['date'] = pd.to_datetime(results['date'])

filtered = results[
    (results['date'].dt.year >= 1990) &
    (results['tournament'] != 'Friendly')
].copy()

print(f'Original dataset: {len(results)} matches')
print(f'Filtered dataset: {len(filtered)} matches')
print(f'Removed: {len(results) - len(filtered)} matches')
print(f'\nDate range: {filtered["date"].min()} to {filtered["date"].max()}')

Original dataset: 49498 matches
Filtered dataset: 21551 matches
Removed: 27947 matches

Date range: 1990-01-29 00:00:00 to 2026-07-06 00:00:00


In [9]:
# What tournaments remain after filtering?
print(filtered['tournament'].value_counts().head(15))


tournament
FIFA World Cup qualification            6977
UEFA Euro qualification                 2114
African Cup of Nations qualification    1865
UEFA Nations League                      658
FIFA World Cup                           645
African Cup of Nations                   642
AFC Asian Cup qualification              632
CFU Caribbean Cup qualification          508
CECAFA Cup                               422
CONCACAF Nations League                  422
Gold Cup                                 420
Island Games                             384
Copa América                             378
COSAFA Cup                               354
UEFA Euro                                323
Name: count, dtype: int64


In [10]:
# Tournaments to drop - low quality or irrelevant competitions
drop_tournaments = [
    'Island Games',
    'CECAFA Cup', 
    'COSAFA Cup',
    'CFU Caribbean Cup qualification',
]

filtered = filtered[~filtered['tournament'].isin(drop_tournaments)].copy()

print(f'Matches after dropping low quality tournaments: {len(filtered)}')
print(f'\nRemaining tournaments:')
print(filtered['tournament'].value_counts().head(20))

Matches after dropping low quality tournaments: 19883

Remaining tournaments:
tournament
FIFA World Cup qualification            6977
UEFA Euro qualification                 2114
African Cup of Nations qualification    1865
UEFA Nations League                      658
FIFA World Cup                           645
African Cup of Nations                   642
AFC Asian Cup qualification              632
CONCACAF Nations League                  422
Gold Cup                                 420
Copa América                             378
UEFA Euro                                323
AFC Asian Cup                            298
AFF Championship                         291
Gulf Cup                                 259
CFU Caribbean Cup                        210
UNCAF Cup                                164
SAFF Cup                                 162
Confederations Cup                       140
EAFF Championship                        130
Asian Games                              120
Name: count

In [11]:
# Create outcome column from home team's perspective
def get_outcome(row):
    if row['home_score'] > row['away_score']:
        return 'W'
    elif row['home_score'] < row['away_score']:
        return 'L'
    else:
        return 'D'

filtered['outcome'] = filtered.apply(get_outcome, axis=1)

# Check the distribution
print(filtered['outcome'].value_counts())
print(f'\nAs percentages:')
print(filtered['outcome'].value_counts(normalize=True).round(3) * 100)

outcome
W    9731
L    5821
D    4331
Name: count, dtype: int64

As percentages:
outcome
W    48.9
L    29.3
D    21.8
Name: proportion, dtype: float64


In [12]:
##  Home Field Advantage Analysis

### Outcome Distribution
- **Win (Home): 48.9%** 
- **Loss (Home): 29.3%**
- **Draw: 21.8%**

Home teams win nearly half of all matches, significantly more 
than away teams at 29.3%. This is a well known phenomenon in 
football called **home field advantage**.

### Why Home Teams Win More
1. **Crowd support** — home fans create pressure on the 
   opposition and can influence referee decisions subconsciously
2. **Travel fatigue** — away teams travel long distances, 
   disrupting sleep, nutrition and preparation
3. **Familiarity** — home teams know the pitch conditions, 
   climate and altitude
4. **Referee bias** — studies show referees award more injury 
   time and are more lenient toward the home side due to 
   crowd pressure

### Implication for Our Model
The `neutral` column in our dataset flags matches played at 
neutral venues. At neutral venues, home advantage should 
disappear — making it an important feature for predicting 
World Cup matches, which are hosted in a single country.

SyntaxError: invalid character '—' (U+2014) (2367136690.py, line 13)

In [13]:
# Does neutral venue affect home advantage?
print('=== Home Win % by Venue Type ===')
print(filtered.groupby('neutral')['outcome'].value_counts(normalize=True).round(3) * 100)

=== Home Win % by Venue Type ===
neutral  outcome
False    W          51.5
         L          27.2
         D          21.3
True     W          43.1
         L          34.1
         D          22.8
Name: proportion, dtype: float64


In [14]:
## 📊 Neutral Venue Confirmation

Data confirms home advantage disappears at neutral venues:

| Venue Type | Home Win% | Away Win% | Gap |
|---|---|---|---|
| Home Ground | 51.5% | 27.2% | 24.3% |
| Neutral | 43.1% | 34.1% | 9.0% |

This validates including `neutral` as a feature in our model.
World Cup matches are mostly neutral — so home advantage is 
significantly reduced compared to regular international football.

SyntaxError: invalid character '—' (U+2014) (3696312757.py, line 11)

In [15]:
# Sort by date - crucial for calculating historical features correctly
filtered = filtered.sort_values('date').reset_index(drop=True)

print(f'Date range confirmed: {filtered["date"].min()} to {filtered["date"].max()}')
print(f'\nFirst 3 matches:')
print(filtered[['date', 'home_team', 'away_team', 'home_score', 'away_score']].head(3))
print(f'\nLast 3 matches:')
print(filtered[['date', 'home_team', 'away_team', 'home_score', 'away_score']].tail(3))

Date range confirmed: 1990-01-29 00:00:00 to 2026-07-06 00:00:00

First 3 matches:
        date  home_team away_team  home_score  away_score
0 1990-01-29   Thailand     Kenya         2.0         1.0
1 1990-02-02   Colombia   Uruguay         0.0         2.0
2 1990-02-02  Indonesia     Kenya         2.0         3.0

Last 3 matches:
            date      home_team away_team  home_score  away_score
19880 2026-07-05         Mexico   England         NaN         NaN
19881 2026-07-05         Brazil    Norway         NaN         NaN
19882 2026-07-06  United States   Belgium         NaN         NaN


In [16]:
def get_team_stats(team, date, df, n=10):
    """
    Calculate team statistics from their last N matches
    before a given date. Only looks backwards - no data leakage.
    
    Returns: dict of features for that team at that point in time
    """
    # Get all matches this team played BEFORE this date
    team_matches = df[
        ((df['home_team'] == team) | (df['away_team'] == team)) &
        (df['date'] < date)
    ].tail(n)  # Take only last N matches
    
    # If team has no history return neutral stats
    if len(team_matches) == 0:
        return {
            'games_played': 0,
            'win_rate': 0.5,
            'draw_rate': 0.2,
            'avg_goals_scored': 1.0,
            'avg_goals_conceded': 1.0,
        }
    
    wins, draws, goals_scored, goals_conceded = 0, 0, 0, 0
    
    for _, match in team_matches.iterrows():
        if match['home_team'] == team:
            gf = match['home_score']  # Goals for
            ga = match['away_score']  # Goals against
        else:
            gf = match['away_score']
            ga = match['home_score']
        
        if gf > ga:
            wins += 1
        elif gf == ga:
            draws += 1
            
        goals_scored += gf
        goals_conceded += ga
    
    n_matches = len(team_matches)
    
    return {
        'games_played': n_matches,
        'win_rate': wins / n_matches,
        'draw_rate': draws / n_matches,
        'avg_goals_scored': goals_scored / n_matches,
        'avg_goals_conceded': goals_conceded / n_matches,
    }

print('✅ Team stats function defined!')

✅ Team stats function defined!


In [17]:
# Test our function on a real team
brazil_stats = get_team_stats(
    team='Brazil',
    date=pd.Timestamp('2026-07-01'),
    df=filtered
)

print('=== Brazil stats before July 2026 ===')
for key, value in brazil_stats.items():
    print(f'{key}: {value:.3f}')

=== Brazil stats before July 2026 ===
games_played: 10.000
win_rate: 0.500
draw_rate: 0.200
avg_goals_scored: 1.600
avg_goals_conceded: 1.000


In [18]:
# Test our function on a real team
argentina_stats = get_team_stats(
    team='Argentina',
    date=pd.Timestamp('2026-07-01'),
    df=filtered
)

print('=== Argentina stats before July 2026 ===')
for key, value in argentina_stats.items():
    print(f'{key}: {value:.3f}')

=== Argentina stats before July 2026 ===
games_played: 10.000
win_rate: 0.800
draw_rate: 0.100
avg_goals_scored: 1.900
avg_goals_conceded: 0.400


In [19]:
# Apply team stats to every match - this builds our feature matrix
# Warning: this may take a minute or two to run

print('Building features for all matches...')
print('This calculates team stats for each of our 19,883 matches')
print('Please wait...\n')

home_stats = []
away_stats = []

for idx, row in filtered.iterrows():
    h = get_team_stats(row['home_team'], row['date'], filtered)
    a = get_team_stats(row['away_team'], row['date'], filtered)
    home_stats.append(h)
    away_stats.append(a)

# Convert to dataframes
home_df = pd.DataFrame(home_stats).add_prefix('home_')
away_df = pd.DataFrame(away_stats).add_prefix('away_')

print('✅ Features built!')
print(f'Home features shape: {home_df.shape}')
print(f'Away features shape: {away_df.shape}')

Building features for all matches...
This calculates team stats for each of our 19,883 matches
Please wait...

✅ Features built!
Home features shape: (19883, 5)
Away features shape: (19883, 5)


In [20]:
# Combine all features into one dataframe
model_df = pd.concat([
    filtered[['date', 'home_team', 'away_team', 
              'tournament', 'neutral', 'outcome']].reset_index(drop=True),
    home_df.reset_index(drop=True),
    away_df.reset_index(drop=True)
], axis=1)

# Drop rows where we have NaN scores (upcoming matches)
model_df = model_df.dropna()

print(f'Final dataset shape: {model_df.shape}')
print(f'\nColumns: {list(model_df.columns)}')
print(f'\nSample row:')
model_df.tail(3)

Final dataset shape: (19883, 16)

Columns: ['date', 'home_team', 'away_team', 'tournament', 'neutral', 'outcome', 'home_games_played', 'home_win_rate', 'home_draw_rate', 'home_avg_goals_scored', 'home_avg_goals_conceded', 'away_games_played', 'away_win_rate', 'away_draw_rate', 'away_avg_goals_scored', 'away_avg_goals_conceded']

Sample row:


,date,home_team,away_team,tournament,neutral,outcome,home_games_played,home_win_rate,home_draw_rate,home_avg_goals_scored,home_avg_goals_conceded,away_games_played,away_win_rate,away_draw_rate,away_avg_goals_scored,away_avg_goals_conceded
19880,2026-07-05,Mexico,England,FIFA World Cup,False,D,10,0.9,0.1,1.8,0.3,10,0.9,0.1,2.5,0.3
19881,2026-07-05,Brazil,Norway,FIFA World Cup,True,D,10,0.5,0.2,1.6,1.0,10,0.9,0.0,3.8,1.1
19882,2026-07-06,United States,Belgium,FIFA World Cup,False,D,10,0.7,0.1,2.3,1.0,10,0.6,0.4,3.3,0.7


In [21]:
# Check these specific matches in the original filtered data
mask = filtered['date'] >= '2026-07-05'
print(filtered[mask][['date', 'home_team', 'away_team', 
                       'home_score', 'away_score']].head(10))

            date      home_team away_team  home_score  away_score
19880 2026-07-05         Mexico   England         NaN         NaN
19881 2026-07-05         Brazil    Norway         NaN         NaN
19882 2026-07-06  United States   Belgium         NaN         NaN


In [22]:
# Fix - drop matches where scores were NaN in original data
valid_mask = filtered['home_score'].notna() & filtered['away_score'].notna()
model_df = model_df[valid_mask.values]

print(f'Shape after removing matches with no scores: {model_df.shape}')
print(f'\nLatest matches now:')
model_df.tail(3)[['date', 'home_team', 'away_team', 'outcome']]

Shape after removing matches with no scores: (19872, 16)

Latest matches now:


,date,home_team,away_team,outcome
19869,2026-07-01,England,DR Congo,W
19870,2026-07-01,Belgium,Senegal,W
19871,2026-07-01,United States,Bosnia and Herzegovina,W
